# Task 2.2 — Crammer-Singer M-SVM on Wine Dataset

In [1]:
# ── Imports & seed ────────────────────────────────────────────────
import numpy as np, random
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE); random.seed(RANDOM_STATE)

from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score


**What this block does:** Fixes `random_state=42` everywhere for reproducibility. `LinearSVC(multi_class='crammer_singer')` is the closest sklearn equivalent to the CS row of Table 1 ($K_1=1, K_2=1, M=I$). *Reference: Table 1 CS row; Definition 1.*

In [2]:
# ── Step 1: Load + preprocess Wine ────────────────────────────────
# Hyperparameters (all defined here at top)
C_VALUE   = 1.0    # = 1/lambda in Eq.1
MAX_ITER  = 5000

wine = load_wine()
X, y = wine.data, wine.target
X_scaled = StandardScaler().fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y)
print(f"Train {X_train.shape}, Test {X_test.shape}, Classes {np.unique(y)}")


Train (142, 13), Test (36, 13), Classes [0 1 2]


**(a)** We normalise the 13 physicochemical features so that the linear margin in the CS formulation is not skewed by feature scale. **(b)** Section 2 — input space $\mathcal{X}\subseteq\mathbb{R}^{13}$; Table 2 evaluation protocol (80/20 split).

In [3]:
# ── Step 2: Train Crammer-Singer M-SVM ────────────────────────────
model = LinearSVC(multi_class='crammer_singer', C=C_VALUE,
                  max_iter=MAX_ITER, random_state=RANDOM_STATE)
model.fit(X_train, y_train)
print("Training complete. Classes:", model.classes_)


Training complete. Classes: [0 1 2]


**(a)** `LinearSVC` with `multi_class='crammer_singer'` directly minimises the CS multi-class hinge loss — a native multi-class objective, not one-vs-rest. C=1.0 maps to $\lambda=1/C=1.0$ in the regularisation term $\frac{\lambda}{2}\overline{\|h\|}_M^2$. **(b)** Table 1 CS row: $K_1=1$, $K_2=1$, $M=I$, $p=1$; Definition 1 (Eq. 1).

In [4]:
# ── Step 3: Evaluate — test error % ───────────────────────────────
y_pred = model.predict(X_test)
acc    = accuracy_score(y_test, y_pred)
err    = (1 - acc) * 100
print(f"Test Accuracy : {acc*100:.2f}%")
print(f"Test Error    : {err:.2f}%")


Test Accuracy : 94.44%
Test Error    : 5.56%


**(a)** Test error % = $(1-\text{accuracy})\times100$, exactly the metric in Table 2 of the paper. **(b)** Table 2 — this metric is used for all model comparisons (e.g. CS on CB513 = 23.63%).

In [5]:
# ── Step 4: Summary ───────────────────────────────────────────────
print("="*52)
print("  Crammer-Singer M-SVM — Wine Dataset")
print("="*52)
print(f"  C (=1/lambda) : {C_VALUE}")
print(f"  Test accuracy : {acc*100:.2f}%")
print(f"  Test error    : {err:.2f}%")
print(f"  Paper CS/CB513: 23.63%  (Table 2, reference)")
print("="*52)


  Crammer-Singer M-SVM — Wine Dataset
  C (=1/lambda) : 1.0
  Test accuracy : 94.44%
  Test error    : 5.56%
  Paper CS/CB513: 23.63%  (Table 2, reference)


## Interpretation
Our test error on Wine is far lower than the paper's 23.63% on CB513. This is expected: Wine is tiny (178 samples), low-dimensional (13 features), and well-separated, so a linear CS boundary finds a clean partition easily. CB513 has 84K protein sequences with high overlap and complex features, making it intrinsically harder. The CS formulation is identical in both cases (Table 1), but dataset difficulty drives the error gap, not the algorithm. *Reference: Table 2; Definition 1; Section 2.*